# Eşik Ayarlaması

👇 Çalışacağınız veri setini görmek için `player_performances.csv` oyuncu veri setini yükleyin.

In [1]:
import pandas as pd

!curl -s https://d32aokrjazspmn.cloudfront.net/materials/ML_Player_performance.csv > data/player_performances.csv

data = pd.read_csv('data/player_performances.csv')

data.head()

,games played,minutes played,points per game,field goals made,field goal attempts,field goal percent,3 point made,3 point attempt,3 point %,free throw made,free throw attempts,free throw %,offensive rebounds,defensive rebounds,rebounds,assists,steals,blocks,turnovers,target_5y
0,36,27.4,7.4,2.6,7.6,34.7,0.5,2.1,25.0,1.6,2.3,69.9,0.7,3.4,4.1,1.9,0.4,0.4,1.3,0
1,35,26.9,7.2,2.0,6.7,29.6,0.7,2.8,23.5,2.6,3.4,76.5,0.5,2.0,2.4,3.7,1.1,0.5,1.6,0
2,74,15.3,5.2,2.0,4.7,42.2,0.4,1.7,24.4,0.9,1.3,67.0,0.5,1.7,2.2,1.0,0.5,0.3,1.0,0
3,58,11.6,5.7,2.3,5.5,42.6,0.1,0.5,22.6,0.9,1.3,68.9,1.0,0.9,1.9,0.8,0.6,0.1,1.0,1
4,48,11.5,4.5,1.6,3.0,52.4,0.0,0.1,0.0,1.3,1.9,67.4,1.0,1.5,2.5,0.3,0.3,0.4,0.8,1


ℹ️ Her gözlem bir oyuncuyu temsil eder ve her sütun performansın bir özelliğidir. Hedef `target_5y`, oyuncunun 5 yıldan az [0] veya 5 yıl ve daha fazla [1] profesyonel kariyere sahip olup olmadığını tanımlar.

# Ön İşleme

👇 Ön işleme için çok fazla zaman harcamamak adına, tüm özellik setini Robust Scale ile ölçeklendirin. Bu uygulama optimal değildir, ancak ön işleme ve/veya modellerin hızla çalıştırılması için kullanılabilir.

Ölçeklendirilmiş özellik setini `X_scaled` olarak kaydedin.

In [4]:
from sklearn.preprocessing import RobustScaler

X = data.drop(columns=['target_5y'])

scaler = RobustScaler()

X_scaled = scaler.fit_transform(X)

### ☑️ Kodunuzu kontrol edin

In [5]:
from nbresult import ChallengeResult

result = ChallengeResult('scaled_features',
                         scaled_features = X_scaled
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/hamza/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /mnt/c/Users/emmo/code/hamza-simsek/S16D3-S-data-threshold/tests
plugins: typeguard-4.4.2, anyio-4.8.0, dash-4.1.0
collecting ... collected 1 item

test_scaled_features.py::TestScaled_features::test_scaled_features PASSED [100%]

============================== 1 passed in 1.09s ===============================


💯 You can commit your code:

git add tests/scaled_features.pickle

git commit -m 'Completed scaled_features step'

git push origin master



# Temel Modelleme

🎯 Görev, %90 garantiyle minimum 5 yıl profesyonel olarak devam edecek oyuncuları tespit etmektir.

👇 Varsayılan bir Lojistik Regresyon modeli antrenörün gereksinimlerini karşılayacak mı? Çapraz doğrulama kullanın ve cevabınızı destekleyen skoru `base_score` değişken adı altında kaydedin.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate

log_cv_results = cross_validate(LogisticRegression(max_iter=1000), X_scaled, data['target_5y'], cv=10, scoring =['precision'])

base_score = log_cv_results['test_precision'].mean()

print(f"Varsayılan modelin ortalama hassasiyet skoru: (Precision) {base_score:.4f}")

Varsayılan modelin ortalama hassasiyet skoru: (Precision) 0.7378


### ☑️ Kodunuzu kontrol edin

In [23]:
from nbresult import ChallengeResult

result = ChallengeResult('base_precision',
                         score = base_score
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/hamza/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /mnt/c/Users/emmo/code/hamza-simsek/S16D3-S-data-threshold/tests
plugins: typeguard-4.4.2, anyio-4.8.0, dash-4.1.0
collecting ... collected 1 item

test_base_precision.py::TestBase_precision::test_precision_score PASSED  [100%]

============================== 1 passed in 0.90s ===============================


💯 You can commit your code:

git add tests/base_precision.pickle

git commit -m 'Completed base_precision step'

git push origin master



# Eşik Ayarlaması

👇 Bir oyuncunun profesyonel olarak 5 yıl veya daha fazla sürmesi için %90 kesinlik garantisi veren karar eşiğini bulun. Eşiği `new_threshold` değişken adı altında kaydedin.

<details>
<summary>💡 İpucu</summary>

- [`cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html) ile çapraz doğrulanmış olasılık tahminleri yapın
    
- Farklı eşiklerde kesinlik skorları oluşturmak için olasılıkları [`precision_recall_curve`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html) içine yerleştirin

- 0.9 kesinliği garanti eden eşiği bulun
      
</details>

In [26]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_recall_curve

y_pred_probas_0, y_pred_probas_1 = cross_val_predict(LogisticRegression(),
                                                     X_scaled, data['target_5y'],
                                                     method = "predict_proba").T


precision, recalls, thresholds = precision_recall_curve(data['target_5y'], y_probas)

df_precision = pd.DataFrame({"precision" : precision[:-1], "threshold": thresholds})

new_threshold = df_precision[df_precision['precision'] >= 0.9]['threshold'].min()

print(f"%90 Kesinlik Garanti Eden Yeni Eşik: {new_threshold:.4f}")

%90 Kesinlik Garanti Eden Yeni Eşik: 0.8667


### ☑️ Kodunuzu kontrol edin

In [27]:
from nbresult import ChallengeResult

result = ChallengeResult('decision_threshold',
                         threshold = new_threshold
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/hamza/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /mnt/c/Users/emmo/code/hamza-simsek/S16D3-S-data-threshold/tests
plugins: typeguard-4.4.2, anyio-4.8.0, dash-4.1.0
collecting ... collected 1 item

test_decision_threshold.py::TestDecision_threshold::test_new_threshold PASSED [100%]

============================== 1 passed in 0.80s ===============================


💯 You can commit your code:

git add tests/decision_threshold.pickle

git commit -m 'Completed decision_threshold step'

git push origin master



# Yeni Eşiği Kullanma

🎯 Antrenör potansiyel olarak ilginç bir oyuncu fark etti, ancak bu oyuncunun minimum 5 yıl profesyonel olarak devam edeceğine dair %90 garantinizi istiyor. Oyuncunun verilerini [buradan](https://wagon-public-datasets.s3.amazonaws.com/Machine%20Learning%20Datasets/ML_New_player.csv) indirin.

In [12]:
new_player = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ML_New_player.csv")

new_player

,games played,minutes played,points per game,field goals made,field goal attempts,field goal percent,3 point made,3 point attempt,3 point %,free throw made,free throw attempts,free throw %,offensive rebounds,defensive rebounds,rebounds,assists,steals,blocks,turnovers
0,80,31.4,14.3,5.9,11.1,52.5,0.0,0.1,11.1,2.6,3.9,65.4,3.0,5.0,8.0,2.4,1.1,0.8,2.2


❓ Oyuncuyu antrenöre tavsiye etmeyi göze alır mısınız? Cevabınızı string olarak `recommendation` değişken adı altında "recommend" veya "not recommend" şeklinde kaydedin.

In [20]:
new_player_scaled = scaler.transform(new_player)

model = LogisticRegression()
model.fit(X_scaled, data['target_5y'])

def custom_predict(X, custom_threshold):
    probs = model.predict_proba(X)
    expensive_probs = probs[:, 1]
    return (expensive_probs > custom_threshold)

custom_prediction = custom_predict(X=new_player_scaled, custom_threshold=new_threshold)[0]
print(custom_prediction)
recommendation = "recommend"

True


### ☑️ Kodunuzu kontrol edin

In [21]:
from nbresult import ChallengeResult

result = ChallengeResult('recommendation',
                         recommendation = recommendation
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/hamza/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /mnt/c/Users/emmo/code/hamza-simsek/S16D3-S-data-threshold/tests
plugins: typeguard-4.4.2, anyio-4.8.0, dash-4.1.0
collecting ... collected 1 item

test_recommendation.py::TestRecommendation::test_recommendation PASSED   [100%]

============================== 1 passed in 0.37s ===============================


💯 You can commit your code:

git add tests/recommendation.pickle

git commit -m 'Completed recommendation step'

git push origin master



# 🏁